# Section 4.4 — Attribute-Order Sensitivity and Slack Analysis

This notebook analyses the 576 completed SAA attribute-order configurations. It reads existing populations and optimisation diagnostics only; it does not invoke synthesis. Macro Census TVD is an equally weighted normalised evaluation metric, whereas total Stage 1 slack is a summed absolute count objective; they are analysed and ranked separately.

## 1. Setup and data discovery

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
from PopSynthesis.analyse.paper_section_4_4 import (
    MAIN_ORDER, HEURISTIC_ORDER, VARIANTS, repository_root, run_analysis
)
root = repository_root(Path.cwd())
root

PosixPath('/Users/bobla/Documents/repos/PopSyn_Monash')

## 2. Load experimental results

In [2]:
result = run_analysis(root)
metrics = result['metrics']
print(f"Analysed {len(metrics)} configurations; outputs: {result['output']}")

Analysed 576 configurations; outputs: /Users/bobla/Documents/repos/PopSyn_Monash/IO/output/full_household_analysis/paper_section_4_4


## 3. Validate permutation coverage and metrics

In [3]:
coverage = metrics.groupby(['variant', 'variant_label']).size().rename('permutations')
display(coverage)
display(result['validations'].groupby('passed').size().rename('checks'))
assert result['validations']['passed'].all()

variant       variant_label   
bn_pool       SAA BN-pool         120
relaxed       SAA Relaxed         120
seed_free     SAA Seed-free       120
seed_only     SAA Seed-only        96
unrestricted  SAA Unrestricted    120
Name: permutations, dtype: int64

passed
True    1760
Name: checks, dtype: int64

## 4. Define ordering strategies and heuristic scores

In [4]:
print('Production order:', ' > '.join(MAIN_ORDER))
print('Unconstrained VISTA heuristic:', ' > '.join(HEURISTIC_ORDER))
display(result['scores'].sort_values(['score_source', 'heuristic_score'], ascending=[True, False]))

Production order: hhsize > dwelltype > owndwell > hhinc > totalvehs
Unconstrained VISTA heuristic: dwelltype > owndwell > hhinc > hhsize > totalvehs


,score_source,attribute,rigidity,marginal_pressure,heuristic_score
7,BN pool,owndwell,0.163477,0.101790,0.265267
6,BN pool,dwelltype,0.160426,0.059551,0.219977
8,BN pool,hhinc,0.026101,0.187972,0.214073
5,BN pool,hhsize,0.086934,0.045421,0.132355
9,BN pool,totalvehs,0.070168,0.049685,0.119853
1,VISTA,dwelltype,0.506785,0.059885,0.566670
2,VISTA,owndwell,0.364973,0.097404,0.462377
3,VISTA,hhinc,0.209756,0.187708,0.397464
0,VISTA,hhsize,0.293546,0.045476,0.339022
4,VISTA,totalvehs,0.284633,0.049603,0.334236


## 5. Figure A: Census–joint fidelity sensitivity

In [5]:
display(Markdown(f"![Figure A]({result['output'] / 'figure_A_order_sensitivity.png'})"))

![Figure A](/Users/bobla/Documents/repos/PopSyn_Monash/IO/output/full_household_analysis/paper_section_4_4/figure_A_order_sensitivity.png)

## 6. Figure B: Ordering outcome distributions

In [6]:
display(Markdown(f"![Figure B]({result['output'] / 'figure_B_order_distributions.png'})"))
display(result['summary'])

![Figure B](/Users/bobla/Documents/repos/PopSyn_Monash/IO/output/full_household_analysis/paper_section_4_4/figure_B_order_distributions.png)

,variant,variant_label,metric,n,minimum,maximum,mean,median,standard_deviation,iqr,p05,p95,exact_count,exact_percent
0,unrestricted,SAA Unrestricted,census_macro_tvd,120,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,120,100.0
1,unrestricted,SAA Unrestricted,total_stage1_slack,120,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,120,100.0
2,unrestricted,SAA Unrestricted,five_way_tvd,120,0.256022,0.294025,0.271951,0.271191,0.007731,0.010117,0.260291,0.285957,0,0.0
3,unrestricted,SAA Unrestricted,five_way_js_distance,120,0.324992,0.356239,0.341728,0.341488,0.006799,0.009733,0.330445,0.352468,0,0.0
4,unrestricted,SAA Unrestricted,mean_pairwise_tvd,120,0.137539,0.144641,0.141002,0.140674,0.001880,0.002945,0.138218,0.144032,0,0.0
5,relaxed,SAA Relaxed,census_macro_tvd,120,0.005713,0.015395,0.010811,0.010445,0.002331,0.003258,0.006898,0.014600,0,0.0
6,relaxed,SAA Relaxed,total_stage1_slack,120,4310.000000,45872.000000,16426.116667,11938.000000,11718.249404,12893.500000,5707.500000,40213.900000,0,0.0
7,relaxed,SAA Relaxed,five_way_tvd,120,0.267858,0.300973,0.282771,0.282236,0.006593,0.008568,0.273152,0.294644,0,0.0
8,relaxed,SAA Relaxed,five_way_js_distance,120,0.323245,0.350050,0.339382,0.340283,0.005648,0.007377,0.328634,0.347486,0,0.0
9,relaxed,SAA Relaxed,mean_pairwise_tvd,120,0.138842,0.146770,0.142565,0.142342,0.001870,0.002616,0.139583,0.145522,0,0.0


## 7. Ordering rankings and selected-strategy comparison

In [7]:
display(pd.read_csv(result['output'] / 'ordering_comparison_main.csv'))
display(result['selected'])

,variant,variant_label,ordering_strategy,overlap_or_tie_note,macro_census_tvd,total_stage1_slack,five_way_js_distance,macro_tvd_rank,joint_rank
0,unrestricted,SAA Unrestricted,Household-size-first; Minimum macro TVD; Minim...,all 120 eligible orders tie,0.000000,0.0,0.338496,1/120,40/120
1,unrestricted,SAA Unrestricted,Minimum five-way JS,NaN,0.000000,0.0,0.324992,1/120,1/120
2,unrestricted,SAA Unrestricted,Unconstrained heuristic,NaN,0.000000,0.0,0.332553,1/120,14/120
3,relaxed,SAA Relaxed,Household-size-first,NaN,0.014322,13070.0,0.334689,110/120,26/120
4,relaxed,SAA Relaxed,Minimum five-way JS,NaN,0.009576,6346.0,0.323245,38/120,1/120
5,relaxed,SAA Relaxed,Minimum macro TVD,NaN,0.005713,5318.0,0.346608,1/120,109/120
6,relaxed,SAA Relaxed,Minimum total slack,NaN,0.007270,4310.0,0.327518,9/120,4/120
7,relaxed,SAA Relaxed,Unconstrained heuristic,NaN,0.009386,6316.0,0.330039,36/120,9/120
8,seed_only,SAA Seed-only,Household-size-first,NaN,0.017678,48446.0,0.329800,72/96,14/96
9,seed_only,SAA Seed-only,Minimum five-way JS,NaN,0.012247,41514.0,0.322455,12/96,1/96


,variant,variant_label,strategy,overlapping_strategies,optimum_tie_count,tie_note,rank_denominator,permutation_index,order_text,census_macro_tvd,...,census_difference_from_median,five_way_tvd,five_way_js_distance,joint_rank,joint_percentile,joint_difference_from_best,joint_difference_from_median,total_stage1_slack,slack_rank,slack_difference_from_best
0,unrestricted,SAA Unrestricted,household_size_first,household_size_first; minimum_macro_tvd; minim...,1,unique,120,10,hhsize > dwelltype > owndwell > hhinc > totalvehs,0.000000,...,0.000000,0.256553,0.338496,40,67.226891,0.013505,-0.002992,0.0,1,0.0
1,unrestricted,SAA Unrestricted,unconstrained_heuristic,unconstrained_heuristic,1,unique,120,65,dwelltype > owndwell > hhinc > hhsize > totalvehs,0.000000,...,0.000000,0.258981,0.332553,14,89.075630,0.007561,-0.008936,0.0,1,0.0
2,unrestricted,SAA Unrestricted,minimum_macro_tvd,household_size_first; minimum_macro_tvd; minim...,120,all 120 eligible orders tie,120,10,hhsize > dwelltype > owndwell > hhinc > totalvehs,0.000000,...,0.000000,0.256553,0.338496,40,67.226891,0.013505,-0.002992,0.0,1,0.0
3,unrestricted,SAA Unrestricted,minimum_total_slack,household_size_first; minimum_macro_tvd; minim...,120,all 120 eligible orders tie,120,10,hhsize > dwelltype > owndwell > hhinc > totalvehs,0.000000,...,0.000000,0.256553,0.338496,40,67.226891,0.013505,-0.002992,0.0,1,0.0
4,unrestricted,SAA Unrestricted,minimum_five_way_js,minimum_five_way_js; median_census,1,unique,120,113,hhinc > dwelltype > owndwell > hhsize > totalvehs,0.000000,...,0.000000,0.263608,0.324992,1,100.000000,0.000000,-0.016496,0.0,1,0.0
5,unrestricted,SAA Unrestricted,median_census,minimum_five_way_js; median_census,1,unique,120,113,hhinc > dwelltype > owndwell > hhsize > totalvehs,0.000000,...,0.000000,0.263608,0.324992,1,100.000000,0.000000,-0.016496,0.0,1,0.0
6,relaxed,SAA Relaxed,household_size_first,household_size_first,1,unique,120,10,hhsize > dwelltype > owndwell > hhinc > totalvehs,0.014322,...,0.003877,0.267858,0.334689,26,78.991597,0.011444,-0.005595,13070.0,67,8760.0
7,relaxed,SAA Relaxed,unconstrained_heuristic,unconstrained_heuristic,1,unique,120,65,dwelltype > owndwell > hhinc > hhsize > totalvehs,0.009386,...,-0.001059,0.268765,0.330039,9,93.277311,0.006794,-0.010244,6316.0,14,2006.0
8,relaxed,SAA Relaxed,minimum_macro_tvd,minimum_macro_tvd,1,unique,120,39,totalvehs > owndwell > dwelltype > hhsize > hhinc,0.005713,...,-0.004732,0.282570,0.346608,109,9.243697,0.023363,0.006324,5318.0,4,1008.0
9,relaxed,SAA Relaxed,minimum_total_slack,minimum_total_slack,1,unique,120,110,hhinc > dwelltype > hhsize > owndwell > totalvehs,0.007270,...,-0.003175,0.279642,0.327518,4,97.478992,0.004273,-0.012766,4310.0,1,0.0


## 8. Extract and validate stage-level slack

In [8]:
print('Slack units: summed absolute household-count discrepancies; values are not normalised TVDs.')
slack = pd.read_csv(result['output'] / 'slack_by_permutation.csv')
display(slack.groupby('variant').agg(total_slack_min=('total_slack','min'), total_slack_median=('total_slack','median'), total_slack_max=('total_slack','max')))

Slack units: summed absolute household-count discrepancies; values are not normalised TVDs.


,total_slack_min,total_slack_median,total_slack_max
variant,,,
bn_pool,1482.0,2026.0,6638.0
relaxed,4310.0,11938.0,45872.0
seed_free,0.0,0.0,0.0
seed_only,39726.0,47401.0,83844.0
unrestricted,0.0,0.0,0.0


## 9. Figure C: Slack by attribute and position

In [9]:
display(Markdown(f"![Figure C]({result['output'] / 'figure_C_slack_by_attribute_position.png'})"))
display(result['slack_position'])

![Figure C](/Users/bobla/Documents/repos/PopSyn_Monash/IO/output/full_household_analysis/paper_section_4_4/figure_C_slack_by_attribute_position.png)

,variant,attribute,position,mean_total_slack,median_total_slack,p95_total_slack,max_total_slack,contributing_permutations,mean_nonzero_zone_percent,mean_positive_slack,mean_negative_slack
0,bn_pool,dwelltype,1,0.000000,0.0,0.0,0.0,24,0.000000,0.000000,0.000000
1,bn_pool,dwelltype,2,0.000000,0.0,0.0,0.0,24,0.000000,0.000000,0.000000
2,bn_pool,dwelltype,3,452.666667,265.0,1912.6,1960.0,24,7.079112,226.333333,226.333333
3,bn_pool,dwelltype,4,1532.833333,1298.0,3265.7,3632.0,24,23.890497,766.416667,766.416667
4,bn_pool,dwelltype,5,4784.833333,4727.0,5498.8,5880.0,24,56.289194,2392.416667,2392.416667
...,...,...,...,...,...,...,...,...,...,...,...
119,unrestricted,totalvehs,1,0.000000,0.0,0.0,0.0,24,0.000000,0.000000,0.000000
120,unrestricted,totalvehs,2,0.000000,0.0,0.0,0.0,24,0.000000,0.000000,0.000000
121,unrestricted,totalvehs,3,0.000000,0.0,0.0,0.0,24,0.000000,0.000000,0.000000
122,unrestricted,totalvehs,4,0.000000,0.0,0.0,0.0,24,0.000000,0.000000,0.000000


## 10. Figure D: Rigidity, pressure, and slack

In [10]:
display(Markdown(f"![Figure D]({result['output'] / 'figure_D_rigidity_pressure_vs_slack.png'})"))
display(result['relationship'])

![Figure D](/Users/bobla/Documents/repos/PopSyn_Monash/IO/output/full_household_analysis/paper_section_4_4/figure_D_rigidity_pressure_vs_slack.png)

,variant,variant_label,score_source,attribute,rigidity,marginal_pressure,heuristic_score,mean_late_attribute_slack,late_attribute_observations,mean_total_slack_early,early_permutations,mean_total_slack_late,late_permutations,early_placement_benefit,spearman_rigidity_late_slack,spearman_pressure_late_slack,spearman_score_early_benefit
0,unrestricted,SAA Unrestricted,VISTA,hhsize,0.293546,0.045476,0.339022,0.000000,48,0.000000,48,0.000000,48,0.000000,NaN,NaN,NaN
1,unrestricted,SAA Unrestricted,VISTA,dwelltype,0.506785,0.059885,0.566670,0.000000,48,0.000000,48,0.000000,48,0.000000,NaN,NaN,NaN
2,unrestricted,SAA Unrestricted,VISTA,owndwell,0.364973,0.097404,0.462377,0.000000,48,0.000000,48,0.000000,48,0.000000,NaN,NaN,NaN
3,unrestricted,SAA Unrestricted,VISTA,hhinc,0.209756,0.187708,0.397464,0.000000,48,0.000000,48,0.000000,48,0.000000,NaN,NaN,NaN
4,unrestricted,SAA Unrestricted,VISTA,totalvehs,0.284633,0.049603,0.334236,0.000000,48,0.000000,48,0.000000,48,0.000000,NaN,NaN,NaN
5,relaxed,SAA Relaxed,VISTA,hhsize,0.293546,0.045476,0.339022,9349.541667,48,17734.583333,48,15811.708333,48,-1922.875000,0.5,-0.1,0.6
6,relaxed,SAA Relaxed,VISTA,dwelltype,0.506785,0.059885,0.566670,19734.583333,48,9412.875000,48,26663.750000,48,17250.875000,0.5,-0.1,0.6
7,relaxed,SAA Relaxed,VISTA,owndwell,0.364973,0.097404,0.462377,2660.750000,48,18615.916667,48,13450.333333,48,-5165.583333,0.5,-0.1,0.6
8,relaxed,SAA Relaxed,VISTA,hhinc,0.209756,0.187708,0.397464,4618.708333,48,17716.333333,48,13410.541667,48,-4305.791667,0.5,-0.1,0.6
9,relaxed,SAA Relaxed,VISTA,totalvehs,0.284633,0.049603,0.334236,1641.791667,48,18650.875000,48,12794.250000,48,-5856.625000,0.5,-0.1,0.6


## 11. Compare ordering heuristics

In [11]:
display(result['strategies'])

,variant,variant_label,score_source,strategy,order_text,eligible,permutation_index,total_stage1_slack,slack_rank,slack_difference_from_best,census_macro_tvd,census_rank,census_difference_from_best,five_way_tvd,five_way_js_distance,joint_rank,joint_difference_from_best
0,unrestricted,SAA Unrestricted,VISTA,descending_H,dwelltype > owndwell > hhinc > hhsize > totalvehs,True,65.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.258981,0.332553,14.0,0.007561
1,unrestricted,SAA Unrestricted,VISTA,descending_R,dwelltype > owndwell > hhsize > totalvehs > hhinc,True,61.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.268581,0.346561,85.0,0.021569
2,unrestricted,SAA Unrestricted,VISTA,descending_D,hhinc > owndwell > dwelltype > totalvehs > hhsize,True,120.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.287298,0.344013,70.0,0.019022
3,unrestricted,SAA Unrestricted,VISTA,household_first_then_H,hhsize > dwelltype > owndwell > hhinc > totalvehs,True,10.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.256553,0.338496,40.0,0.013505
4,unrestricted,SAA Unrestricted,VISTA,best_macro_tvd,hhsize > dwelltype > owndwell > hhinc > totalvehs,True,10.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.256553,0.338496,40.0,0.013505
5,unrestricted,SAA Unrestricted,VISTA,best_total_slack,hhsize > dwelltype > owndwell > hhinc > totalvehs,True,10.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.256553,0.338496,40.0,0.013505
6,unrestricted,SAA Unrestricted,VISTA,best_joint,hhinc > dwelltype > owndwell > hhsize > totalvehs,True,113.0,0.0,1.0,0.0,0.000000,1.0,0.000000,0.263608,0.324992,1.0,0.000000
7,relaxed,SAA Relaxed,VISTA,descending_H,dwelltype > owndwell > hhinc > hhsize > totalvehs,True,65.0,6316.0,14.0,2006.0,0.009386,36.0,0.003674,0.268765,0.330039,9.0,0.006794
8,relaxed,SAA Relaxed,VISTA,descending_R,dwelltype > owndwell > hhsize > totalvehs > hhinc,True,61.0,9384.0,44.0,5074.0,0.011402,68.0,0.005689,0.276097,0.342610,83.0,0.019365
9,relaxed,SAA Relaxed,VISTA,descending_D,hhinc > owndwell > dwelltype > totalvehs > hhsize,True,120.0,12640.0,66.0,8330.0,0.011967,80.0,0.006254,0.296408,0.340467,62.0,0.017222


## 12. Generate publication tables

In [12]:
for name in ['ordering_comparison_main.tex', 'ordering_comparison_table.tex', 'ordering_sensitivity_table.tex', 'heuristic_slack_table.tex']:
    print(name, (result['output'] / name).stat().st_size, 'bytes')

ordering_comparison_main.tex 2241 bytes
ordering_comparison_table.tex 3900 bytes
ordering_sensitivity_table.tex 566 bytes
heuristic_slack_table.tex 4997 bytes


## 13. Final consistency checks

In [13]:
required = ['permutation_metrics.csv', 'ordering_sensitivity_summary.csv', 'selected_order_comparison.csv', 'ordering_rankings.csv', 'heuristic_scores.csv', 'slack_by_permutation.csv', 'slack_by_attribute.csv', 'slack_by_position.csv', 'slack_rigidity_relationship.csv', 'heuristic_strategy_comparison.csv', 'ordering_comparison_main.csv', 'ordering_comparison_detailed.csv', 'ordering_comparison_main.tex', 'section_4_4_findings.md']
missing = [name for name in required if not (result['output'] / name).exists()]
assert not missing, missing
print('All requested tables, validation records, figures, and findings are present.')

All requested tables, validation records, figures, and findings are present.


## 14. Summarise manuscript findings

In [14]:
display(Markdown((result['output'] / 'section_4_4_findings.md').read_text()))

# Section 4.4 — Attribute-Order Sensitivity and Slack Analysis

All reported variation is variation across attribute-order configurations, not stochastic run-to-run variation.

## A. Coverage and validation

- SAA Unrestricted: 120 eligible permutations.
- SAA Relaxed: 120 eligible permutations.
- SAA Seed-only: 96 eligible permutations.
- SAA Seed-free: 120 eligible permutations.
- SAA BN-pool: 120 eligible permutations.
- Total coverage is 576 configurations. VISTA Seed-only contains 96 rather than 120 orders because its 24 `hhinc`-first permutations are excluded: Negative income has no VISTA-pool support.
- 1760/1760 validation checks pass. Stage 1 slack is nonnegative, every first attribute has zero slack, and summed Stage 1 slack equals final Census L1 for every permutation.
- Slack values are summed absolute household counts over zones, controlled attributes, and states. Macro Census TVD instead equally averages normalised zone-attribute discrepancies over 691 POAs and five attributes; the two measures are not interchangeable.

## B. Ordering sensitivity

- SAA Unrestricted: macro TVD min/median/max/SD = 0.000000/0.000000/0.000000/0.000000; total slack = 0/0/0/0.0; five-way JS = 0.324992/0.341488/0.356239/0.006799.
  - Minimum macro TVD: `hhsize > dwelltype > owndwell > hhinc > totalvehs`; TVD=0.000000, slack=0, JS=0.338496 (all 120 eligible orders tie).
  - Minimum total slack: `hhsize > dwelltype > owndwell > hhinc > totalvehs`; TVD=0.000000, slack=0, JS=0.338496 (all 120 eligible orders tie).
  - Minimum five-way JS: `hhinc > dwelltype > owndwell > hhsize > totalvehs`; TVD=0.000000, slack=0, JS=0.324992 (unique).
- SAA Relaxed: macro TVD min/median/max/SD = 0.005713/0.010445/0.015395/0.002331; total slack = 4,310/11,938/45,872/11,718.2; five-way JS = 0.323245/0.340283/0.350050/0.005648.
  - Minimum macro TVD: `totalvehs > owndwell > dwelltype > hhsize > hhinc`; TVD=0.005713, slack=5,318, JS=0.346608 (unique).
  - Minimum total slack: `hhinc > dwelltype > hhsize > owndwell > totalvehs`; TVD=0.007270, slack=4,310, JS=0.327518 (unique).
  - Minimum five-way JS: `hhinc > dwelltype > owndwell > hhsize > totalvehs`; TVD=0.009576, slack=6,346, JS=0.323245 (unique).
- SAA Seed-only: macro TVD min/median/max/SD = 0.010599/0.014863/0.020006/0.002508; total slack = 39,726/47,401/83,844/11,793.5; five-way JS = 0.322455/0.335428/0.346168/0.004688.
  - Minimum macro TVD: `dwelltype > hhinc > hhsize > owndwell > totalvehs`; TVD=0.010599, slack=39,726, JS=0.331243 (unique).
  - Minimum total slack: `dwelltype > hhinc > hhsize > owndwell > totalvehs`; TVD=0.010599, slack=39,726, JS=0.331243 (unique).
  - Minimum five-way JS: `owndwell > dwelltype > hhinc > hhsize > totalvehs`; TVD=0.012247, slack=41,514, JS=0.322455 (unique).
- SAA Seed-free: macro TVD min/median/max/SD = 0.000000/0.000000/0.000000/0.000000; total slack = 0/0/0/0.0; five-way JS = 0.637852/0.699574/0.779487/0.036781.
  - Minimum macro TVD: `hhsize > dwelltype > owndwell > hhinc > totalvehs`; TVD=0.000000, slack=0, JS=0.661720 (all 120 eligible orders tie).
  - Minimum total slack: `hhsize > dwelltype > owndwell > hhinc > totalvehs`; TVD=0.000000, slack=0, JS=0.661720 (all 120 eligible orders tie).
  - Minimum five-way JS: `dwelltype > hhinc > hhsize > owndwell > totalvehs`; TVD=0.000000, slack=0, JS=0.637852 (unique).
- SAA BN-pool: macro TVD min/median/max/SD = 0.002952/0.003751/0.005692/0.000782; total slack = 1,482/2,026/6,638/1,507.3; five-way JS = 0.423208/0.429657/0.437259/0.003020.
  - Minimum macro TVD: `hhsize > hhinc > totalvehs > dwelltype > owndwell`; TVD=0.002952, slack=2,428, JS=0.432552 (unique).
  - Minimum total slack: `hhsize > totalvehs > dwelltype > hhinc > owndwell`; TVD=0.003044, slack=1,482, JS=0.430237 (unique).
  - Minimum five-way JS: `owndwell > hhsize > hhinc > totalvehs > dwelltype`; TVD=0.005598, slack=6,064, JS=0.423208 (unique).
- By across-order macro-TVD standard deviation, SAA Seed-only is most sensitive (0.002508). Figure A shows that the TVD-, slack-, and JS-minimising orders need not coincide.

## C. Selected household-size-first ordering

- SAA Unrestricted: macro TVD 0.000000 (rank 1/120); total slack 0 (rank 1/120); five-way JS 0.338496 (rank 40/120).
- SAA Relaxed: macro TVD 0.014322 (rank 110/120); total slack 13,070 (rank 67/120); five-way JS 0.334689 (rank 26/120).
- SAA Seed-only: macro TVD 0.017678 (rank 72/96); total slack 48,446 (rank 52/96); five-way JS 0.329800 (rank 14/96).
- SAA Seed-free: macro TVD 0.000000 (rank 1/120); total slack 0 (rank 1/120); five-way JS 0.661720 (rank 22/120).
- SAA BN-pool: macro TVD 0.003511 (rank 36/120); total slack 1,698 (rank 16/120); five-way JS 0.426269 (rank 16/120).
The production order is a policy decision: introducing `hhsize` first guarantees exact household-size marginals at initialization. It is not an empirically selected global optimum. Notably, it ranks 110/120 on macro TVD for SAA Relaxed despite ranking 26/120 on five-way JS.

## D. Heuristic effectiveness

- SAA Unrestricted: household-first TVD/slack/JS = 0.000000/0/0.338496 (ranks 1/1/40 of 120); unconstrained heuristic = 0.000000/0/0.332553 (ranks 1/1/14 of 120).
- SAA Relaxed: household-first TVD/slack/JS = 0.014322/13,070/0.334689 (ranks 110/67/26 of 120); unconstrained heuristic = 0.009386/6,316/0.330039 (ranks 36/14/9 of 120).
- SAA Seed-only: household-first TVD/slack/JS = 0.017678/48,446/0.329800 (ranks 72/52/14 of 96); unconstrained heuristic = 0.012427/41,690/0.325206 (ranks 16/8/5 of 96).
- SAA Seed-free: household-first TVD/slack/JS = 0.000000/0/0.661720 (ranks 1/1/22 of 120); unconstrained heuristic = 0.000000/0/0.662718 (ranks 1/1/25 of 120).
- SAA BN-pool: household-first TVD/slack/JS = 0.003511/1,698/0.426269 (ranks 36/16/16 of 120); unconstrained heuristic = 0.004196/1,974/0.428459 (ranks 79/53/45 of 120).
- `heuristic_strategy_comparison.csv` compares descending H, R-only, D-only, household-first-plus-H, and the three empirical minima. The results are mixed: no input-derived rule dominates every support policy or every objective.
- BN-pool uses BN-specific rigidity and pressure scores; VISTA-derived rigidity is not silently transferred to that pool.

## E. Marginal slack

- SAA Unrestricted: all attributes at all positions have zero Stage 1 slack.
- SAA Relaxed: largest mean attribute-stage slack occurs for Dwelling type at position 5 (31,795.8; n=24).
- SAA Seed-only: largest mean attribute-stage slack occurs for Household income at position 5 (40,618.8; n=24).
- SAA Seed-free: all attributes at all positions have zero Stage 1 slack.
- SAA BN-pool: largest mean attribute-stage slack occurs for Dwelling type at position 5 (4,784.8; n=24).
- Figure C reports mean summed absolute count slack. Position 1 is zero by construction; zero first-position slack does not make the full ordering optimal. The VISTA Seed-only `hhinc`-first cell is N/A, not zero.
- The restrictions are empirical pool-support restrictions and must not be interpreted as verified structural zeros.

## F. Rigidity and marginal pressure

- SAA Relaxed: descriptive Spearman R–late-slack = 0.500, D–late-slack = -0.100, and H–early-benefit = 0.600.
- SAA Seed-only: descriptive Spearman R–late-slack = -0.100, D–late-slack = 0.500, and H–early-benefit = 0.100.
- SAA BN-pool: descriptive Spearman R–late-slack = 0.600, D–late-slack = -0.200, and H–early-benefit = -0.300.
- Correlations are undefined for Unrestricted and Seed-free because slack is identically zero; they are not displayed in Figure D.
- With only five attributes per policy, the mixed signs are descriptive and do not validate the heuristic statistically or causally. In particular, BN-pool has H–early-benefit Spearman correlation -0.300.

## G. Manuscript interpretation

1. All 120 Unrestricted and Seed-free orders attain exact Census marginals, but their joint fidelity still varies (Figure A), so order remains relevant even when slack is zero.
2. VISTA Seed-only is most macro-TVD-sensitive: 0.010599–0.020006, with SD 0.002508 across 96 eligible orders (Figure B).
3. Relaxed SAA separates the objectives clearly: minimum macro TVD has slack 5,318, whereas minimum total slack is 4,310 and occurs under another order (Figure A; Table `ordering_comparison_main.tex`).
4. BN-pool shows the same distinction: the minimum-macro-TVD order has slack 2,428, while the minimum-slack order achieves 1,482.
5. The production order trades Census accuracy for policy priority and relatively strong joint fidelity under Relaxed SAA: macro-TVD rank 110/120 versus JS rank 26/120.
6. Figure C shows that late dwelling type and household income adjustments dominate slack under restricted support, but the responsible attribute differs by policy.
7. The rigidity/pressure evidence is mixed rather than confirmatory; no heuristic strategy dominates every policy and objective (Figure D and `heuristic_strategy_comparison.csv`).

## H. Limitations

- Ordering sensitivity is evaluated on one case study.
- Within-dataset permutation sensitivity is not independent validation of the heuristic.
- Total Stage 1 slack is an absolute count objective; macro Census TVD is an equally weighted normalised evaluation metric.
- Fidelity to weighted VISTA is reference-joint fidelity, not evidence of population realism.
- Empirical pool-support restrictions are not verified structural constraints.
- Permutation variability is configuration sensitivity, not stochastic sampling variability.
- The input-derived heuristic may perform differently under other datasets, pools, or support policies.

Validation status: 1760/1760 checks passed.
